# Tracing & Observability

*Notebook 25*

The OpenAI Agents SDK captures a complete trace of every agent run automatically. This notebook shows you how to read those traces, what to look for when debugging, and how to use them to monitor production agents.
<br>
<br>

**Topics:**
- What tracing captures automatically — no setup required

- Running agents that produce traces worth reading

- Guided trace inspection — what to look for at each step

- Debugging bad runs systematically — connecting reliability, evaluation, and observability

- Comparing latency and token usage across model choices

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner, function_tool

# Notebook-specific imports
import time

MODEL = "gpt-5-mini"
REASONING_MODEL = "gpt-5"


print("✅ Ready!")

---

## 🔭 What Tracing Captures

Every `Runner.run()` call creates a trace automatically. Each trace contains:

- The full input and output of the run
- Every tool call — name, arguments, return value, and duration
- Every model call — the exact prompt sent and response received
- Token usage at each step (input tokens, output tokens)
- Total latency and per-step latency
- Handoffs between agents in multi-agent runs — handoff spans show which agent took over and when (patterns introduced in Lesson 14)
- Errors and the exact step where they occurred

A token is roughly a word or subword chunk — model pricing and context limits are based on tokens.

---

## 🔬 Part 1: Generating Traces Worth Reading

We need an agent with enough structure to make the trace interesting.

In [ ]:
INVENTORY = {
    "laptop": {"price": 1299.99, "stock": 5, "category": "electronics"},
    "headphones": {"price": 149.99, "stock": 23, "category": "electronics"},
    "notebook": {"price": 4.99, "stock": 200, "category": "stationery"},
    "desk lamp": {"price": 39.99, "stock": 0, "category": "electronics"}
}


@function_tool
def search_inventory(query: str) -> str:
    """Search inventory for products matching a query string."""
    matches = [
        f"{name}: ${item['price']:.2f} — category: {item.get('category', 'general')}"
        for name, item in INVENTORY.items()
        if query.lower() in name.lower() or query.lower() in item["category"]
    ]
    return "\n".join(matches) if matches else f"No products found for '{query}'"


@function_tool
def check_stock(product_name: str) -> str:
    """Check the exact stock quantity for a specific named product."""
    key = product_name.lower()
    product = INVENTORY.get(key)
    if not product and key.endswith("s"):
        product = INVENTORY.get(key[:-1])
        if product:
            key = key[:-1]
    if not product:
        return f"Product '{product_name}' not found in inventory"
    status = "In stock" if product["stock"] > 0 else "Out of stock"
    return f"{key}: {product['stock']} units — {status}"


@function_tool
def get_category_report(category: str) -> str:
    """Get a full inventory report for a product category."""
    items = [
        (name, item) for name, item in INVENTORY.items()
        if item["category"] == category.lower()
    ]
    if not items:
        return f"No products in category '{category}'"
    total_value = sum(item["price"] * item["stock"] for _, item in items)
    lines = [f"  {name}: ${item['price']:.2f} × {item['stock']} units" for name, item in items]
    return f"Category: {category}\n" + "\n".join(lines) + f"\nTotal inventory value: ${total_value:.2f}"


inventory_instructions = (
    "You are an inventory management assistant.\n"
    "- Use search_inventory to find products by name or category\n"
    "- Use check_stock to get exact stock quantities for specific products\n"
    "- Use get_category_report for full category summaries\n"
    "Always use the most specific tool available for each question."
)

inventory_agent = Agent(
    name="InventoryAssistant",
    instructions=inventory_instructions,
    model=MODEL,
    tools=[search_inventory, check_stock, get_category_report]
)

# --------------------------------------------------------------
print("✅ Inventory agent ready")
print("    Tools: search_inventory | check_stock | get_category_report")

Open the OpenAI traces dashboard at `https://platform.openai.com/traces` in a separate tab. Every `Runner.run()` you execute appears there within a few seconds, tagged by agent name. Keep this tab open as you work through the lesson.

In [ ]:
print("🔬 RUN 1: Focused question — should use check_stock")
print("=" * 60)

start = time.time()

result = await Runner.run(inventory_agent, input="How many laptops do we have in stock right now?")

elapsed = time.time() - start

print(f"Response: {result.final_output}")
print(f"\n⏱️  {elapsed:.1f}s")
print()
print("📍 TRACE INSPECTION CHECKLIST for Run 1:")
print("    □ Find this trace: agent = 'InventoryAssistant', most recent")
print("    □ How many tool call spans are there? (expect: 1)")
print("    □ Which tool was called? (expect: check_stock)")
print("    □ What argument was passed? (expect: 'laptop' or similar)")
print("    □ What did the tool return? (expect: '5 units — In stock')")
print("    □ What is the total token count?")

#### Run 2: A Broader Query

A broader question that may use multiple tools.

In [ ]:
print("🔬 RUN 2: Broad question — may produce one or two tool calls")
print("=" * 60)

start = time.time()

result = await Runner.run(inventory_agent, input="Give me a full electronics report and flag anything out of stock.")

elapsed = time.time() - start

print(f"Response: {result.final_output}")
print(f"\n⏱️  {elapsed:.1f}s")
print()
print("📍 TRACE INSPECTION CHECKLIST for Run 2:")
print("    □ How many tool call spans are there? (expect: 1 or 2 — get_category_report alone is valid; the agent may also follow up with check_stock)")
print("    □ Which tools were called and in what order?")
print("    □ Did the agent use get_category_report for the full report?")
print("    □ Did it rely on get_category_report alone, or did it also call check_stock?")
print("    □ If it used only get_category_report, was that enough to answer correctly?")
print("    □ Compare total tokens with Run 1 — which used more and why?")


### 💡 What These Traces Show

Run 1 shows a clean single-tool trace. Run 2 may involve one or two tool calls. In the dashboard, look for: how many model-call (LLM) spans fired, which tools were called and in what order, and the token count difference vs Run 1. The exact tool count matters less than practicing how to read the trace structure.

---

## 🐛 Part 2: Debugging a Bad Run

The real value of tracing is diagnosing failures. Here's a broken agent — the output is clearly wrong. Use the trace to find exactly why.

In [ ]:
# This agent has a deliberate instruction bug:
# It's told to use search_inventory for EVERYTHING —
# including questions where check_stock would give a precise answer.
# search_inventory returns fuzzy results; check_stock returns exact counts.

buggy_instructions = (
    "You are an inventory assistant.\n"
    "For ALL questions — including specific stock counts — use search_inventory.\n"
    "Do not use check_stock or get_category_report under any circumstances."
)

buggy_agent = Agent(
    name="BuggyInventoryAgent",
    instructions=buggy_instructions,
    model=MODEL,
    tools=[search_inventory, check_stock, get_category_report]
)

# --------------------------------------------------------------
print("✅ Buggy agent ready")
print("    Bug: forced to use search_inventory for everything")
print("    This will produce imprecise answers for stock count questions")

#### Run Buggy Agent

In [ ]:
print("🐛 DEBUGGING DEMO — Broken Agent")
print("=" * 60)

# Ask for an exact stock count — the agent should use check_stock
# but its buggy instructions force it to use search_inventory instead

broken_result = await Runner.run(buggy_agent, input="I need the exact stock count for laptops — how many units do we have?")

print(f"Buggy agent response:")
print(broken_result.final_output)

#### Run Correct Agent

In [ ]:
# Now run the correct agent on the same question for comparison

correct_result = await Runner.run(inventory_agent, input="I need the exact stock count for laptops — how many units do we have?")

print(f"Correct agent response:")
print(correct_result.final_output)
print()
print("=" * 60)
print("📍 DEBUGGING CHECKLIST:")
print("    □ Find both traces — 'BuggyInventoryAgent' vs 'InventoryAssistant'")
print("    □ Buggy trace: which tool was called? (will be search_inventory)")
print("    □ Buggy trace: what did search_inventory return?")
print("    □ Correct trace: which tool was called? (should be check_stock)")
print("    □ What exact difference in tool output explains the response difference?")
print()
print("    The fix: update instructions to say 'use check_stock for exact counts'")
print("    This is how you use a trace to go from symptom → root cause → fix.")


In your own projects, open the trace for any failed eval case (Lesson 09) or guardrail violation first — it turns the symptom into a concrete root cause. The diagnostic question is always the same: was the failure in tool selection, tool arguments, tool output, or the instructions that drove them?

## 📈 Part 3: Comparing Latency and Token Usage

So far we've used traces to explain wrong behavior; now we'll use the same trace data for a different job — comparing cost, latency, and model choice.

Run the same query on `MODEL` and `REASONING_MODEL`. The trace shows the real cost and latency difference — use this to make informed model selection decisions.

This is the reusable pattern for your own projects: hold the task constant, change one variable at a time (model, instructions, or tool set), then compare the traces.

---

## 📈 Part 3: Comparing Latency and Token Usage

Run the same query on `MODEL` and `REASONING_MODEL`. The trace shows the real cost and latency difference — use this to make informed model selection decisions.

In [ ]:
# Same agent, same tools, same query — different model

analysis_instructions = (
    "You are an inventory analyst.\n"
    "Analyze inventory data and make restocking recommendations.\n"
    "Consider stock levels, prices, and categories in your analysis."
)

analysis_agent_mini = Agent(
    name="InventoryAnalyst_mini",
    instructions=analysis_instructions,
    model=MODEL,
    tools=[search_inventory, check_stock, get_category_report]
)

analysis_agent_full = Agent(
    name="InventoryAnalyst_full",
    instructions=analysis_instructions,
    model=REASONING_MODEL,
    tools=[search_inventory, check_stock, get_category_report]
)

# --------------------------------------------------------------
print("✅ Two analyst agents ready")
print(f"    Mini:  {MODEL}")
print(f"    Full:  {REASONING_MODEL}")

#### Same Query, Two Models

In [ ]:
analysis_query = (
    "Review all inventory categories. Identify which products need restocking "
    "and prioritize them by urgency. Explain your reasoning."
)

print("📈 MODEL COMPARISON")
print("=" * 60)
print(f"Query: {analysis_query}\n")

for label, agent in [(f"MODEL ({MODEL})", analysis_agent_mini),
                      (f"REASONING_MODEL ({REASONING_MODEL})", analysis_agent_full)]:
    start = time.time()

    result = await Runner.run(agent, input=analysis_query)

    elapsed = time.time() - start

    print(f"\n--- {label} ---")
    print(f"Time: {elapsed:.1f}s")
    print(f"Response ({len(result.final_output)} chars):")
    print(result.final_output)

print("\n" + "=" * 60)
print("📍 COMPARISON CHECKLIST (check both traces in the dashboard):")
print("    □ Which model took longer? By how much?")
print("    □ Which used more input tokens? Output tokens?")
print("    □ Which produced the more actionable recommendation?")
print("    □ For this task, is the quality difference worth the cost difference?")

### 💡 Why This Matters

Token counts in the trace are the ground truth for cost estimation. Latency in the trace is the ground truth for performance. Running the same query on both models and comparing traces is how you make the `MODEL` vs `REASONING_MODEL` decision with data rather than intuition.

In production, these same metrics tell you when something regressed — a spike in token usage or latency compared to yesterday's baseline is a signal worth investigating before users notice.

---

## 💪 Practice Exercises

### Exercise 1: Trace-Driven Debug

*Covers: tracing — diagnosing bugs from trace output*

A broken agent is provided. Run it, find the bug using only the trace, fix it, and confirm the fix.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Trace-Driven Debug
# --------------------------------------------------------------
# Objective: Use the dashboard trace to find and fix a bug.

# This agent has a bug that causes incorrect category reports.
# The bug is in its instructions — NOT in the tools.

category_broken_instructions = (
    "You are an inventory assistant.\n"
    "When asked about a category, use search_inventory with the category name.\n"
    "Do not use get_category_report."
)

category_agent_broken = Agent(
    name="CategoryAgent_broken",
    instructions=category_broken_instructions,
    model=MODEL,
    tools=[search_inventory, check_stock, get_category_report]
)

# TODO 1: Run this agent with: "Give me a full report on the electronics category"
#           including total inventory value."

# TODO 2: Open the trace and answer:
#           a) Which tool did it call?
#           b) What did that tool return?
#           c) Did the response include total inventory value? (it shouldn't)
#           d) Which tool SHOULD have been called?

# TODO 3: Create a fixed version of the agent with corrected instructions
#           Run the same query and confirm:
#           - The trace shows get_category_report was called
#           - The response includes total inventory value

# --- Write your code below this line ---

## 🎯 Key Takeaways

**Tracing is automatic:**

- Every `Runner.run()` call creates a trace — no configuration needed

- The OpenAI dashboard shows recent traces for your runs

- Each trace is a complete audit trail of every decision the agent made
<br>
<br>

**Use traces to debug:**

- Start with the symptom (wrong output), then trace backward through tool calls

- Check tool arguments first — wrong input to a correct tool is a common failure

- Check tool outputs next — the tool may have returned something unexpected

- Check the model-call (LLM) span last — see exactly what prompt the model saw
<br>
<br>

**Use traces to monitor and decide:**

- Token counts are the ground truth for cost estimation

- Per-step latency shows which step is the bottleneck

- More tool calls than expected usually points to an instruction problem

- Run the same query on `MODEL` vs `REASONING_MODEL` and compare traces before committing to either

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Your Own Latency Experiment
# --------------------------------------------------------------
# Objective: Use traces to answer a question about agent performance.

# Choose ONE of these experiments:

# Option A: Simple vs complex query
#     Run inventory_agent on a simple question and a complex question
#     Compare: tool call count, total tokens, latency
#     Question to answer: does query complexity linearly affect latency?

# Option B: 1 tool vs 3 tools in instructions
#     Create two agents — one with only check_stock, one with all 3 tools
#     Run both on: "How many laptops do we have?"
#     Compare: does having more tools available slow down simple queries?

# Option C: Your own question
#     Design an experiment that uses traces to answer something you're curious about

# TODO 1: Choose an option and set up your experiment
# TODO 2: Run both conditions
# TODO 3: Compare the traces in the dashboard
# TODO 4: Write your findings as comments — what did you learn?

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Tracing is automatic:**

- Every `Runner.run()` call creates a trace — no configuration needed

- The OpenAI dashboard shows recent traces for your runs

- Each trace is a complete audit trail of every decision the agent made
<br>
<br>

**Use traces to debug:**

- Start with the symptom (wrong output), then trace backward through tool calls

- Check tool arguments first — wrong input to a correct tool is a common failure

- Check tool outputs next — the tool may have returned something unexpected

- Check the LLM span last — see exactly what prompt the model saw
<br>
<br>

**Use traces to monitor and decide:**

- Token counts are the ground truth for cost estimation

- Per-step latency shows which step is the bottleneck

- More tool calls than expected usually points to an instruction problem

- Run the same query on `MODEL` vs `REASONING_MODEL` and compare traces before committing to either

---

## 📍 Next Step

**Notebook 26: Capstone #3**  

Build a multi-tool agent with sessions, guardrails, human-in-the-loop approval, and tracing.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-25-tracing-and-observability)

---